# M19: Hybrid AI test — similarity search + a metadata eligibility rule

**Question from the handout:** does making source eligibility explicit improve our evidence, or just encode a brittle rule?

**The one job we test:** a new charge arrives. Before the app uses a stored label as evidence ("you called this store X last time"), it has to pick **which stored label it's allowed to trust.** We call the store of past labels the **label memory**.

### Why this matters for our project

In M15 the book was clean: every past label was one the student had checked. A real app's label memory won't be. It will also hold:

- **old labels the student later changed** (stale),
- **labels from someone else's account** — e.g. a roommate on a shared household view (wrong scope),
- **labels nobody checked**: the bank's own auto-categories, or LLM guesses (missing or weak authority).

Plain similarity search can't tell these apart. Two records with the same store text get the *same* score, whether the student confirmed one or not. So we add a **hybrid** step:

```
new charge ──► similarity search over ALL records ──► eligibility rule on each record's metadata ──► top eligible record
               (the "AI" part, same as M15)            (the "explicit semantics" part)              (or "ask the student")
```

That's the "hybrid": a learned/statistical part (similarity) plus a symbolic part (a written rule over metadata fields).

### Real vs simulated

- **Real:** 211 student-confirmed labels (our hand labels, July–September), the similarity method, and the rule.
- **Simulated (stress-test records we added on purpose):** 3 extra records and 1 broken metadata field, listed in Step 2. The 5 test charges are new October charges, made up in the same bank formats as the rest of our (also made-up) data.

**How to run:** in Colab, upload `transactions.csv` (and optionally `B1_small.txt` + `september_truth.csv` for the last section) when asked, or keep the `data` folder next to this notebook. Then Runtime → Run all.

In [1]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)


def find(name):
    for p in [f"data/{name}", name]:
        if os.path.exists(p):
            return p
    return None


if find("transactions.csv") is None:
    try:
        from google.colab import files  # only exists on Colab
        print("Please upload transactions.csv (and optionally B1_small.txt and september_truth.csv)")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put transactions.csv in a 'data' folder next to this notebook.")

df = pd.read_csv(find("transactions.csv"))
print(len(df), "real labeled charges")

ModuleNotFoundError: No module named 'sklearn'

## Step 1. Same text cleanup as A2 and M15

In [ ]:
PREFIXES = [r"^POS PURCHASE ", r"^CHECKCARD \d{4} ", r"^PURCHASE AUTHORIZED ON \d\d/\d\d "]


def normalize(desc: str) -> str:
    s = desc.upper()
    for p in PREFIXES:
        s = re.sub(p, "", s)
    s = re.sub(r" CARD \d{4}$", "", s)
    s = re.sub(r"(PPD|WEB) ID: ?\d+", "", s)
    s = re.sub(r"REF #\S+", "", s)
    s = s.replace("*", " * ")
    tokens = [t for t in s.split() if not re.search(r"\d", t)]
    return re.sub(r"\s+", " ", " ".join(tokens)).strip()

## Step 2. Build the label memory, with metadata on every record

Every record gets the metadata fields the handout asks for:

| Field | Meaning in our app | Values we use |
|---|---|---|
| `source_id` | permanent ID | `L-049` = the student's label on charge #49; other prefixes = stress-test records |
| `status` | is this label still in force? | `current`, `superseded` (the student changed it later) |
| `authority` | who decided the label? | `student_confirmed`, `bank_auto` (bank's own category), *blank* = unknown |
| `scope` | whose money is this? | `student`, `roommate` |
| `effective_date` | when the label was saved | date |
| `owner` | who can fix this record's fields | the student, the roommate, or nobody (bank import) |

**The real records:** all 211 hand labels become `L-` records with `status=current`, `authority=student_confirmed`, `scope=student`, `owner=student`.

**The stress-test records we inject** (each one is a realistic thing a spending app would end up storing):

| ID | What it is | Why it's tempting |
|---|---|---|
| `RM-001` | Roommate's Venmo payment, labeled **Rent & Utilities** (the roommate pays their rent share by Venmo) | Same text as the student's Venmo charges, saved first (July 1) |
| `V-056a` | The *first* version of the student's label on charge #56 (Amazon Prime): **Shopping**. The student changed it to Subscriptions on 7/30, which created `L-056`. | Same text as every Amazon Prime charge, saved before the fix |
| `BK-039` | The bank's auto-category for charge #39 (Planet Fitness): **Fees** (because the text says "CLUB FEES"). Authority field left blank by the importer. | Saved at import time, before the student reviewed it |
| `L-199` (real record, broken field) | The student's real label for Hudson Valley Urgent Care, **Health** — but saved through an old import path that left `scope` **blank** | It's actually a good record. This tests over-filtering. |

**Order matters:** records are stored in the order they were saved. When two records tie on similarity, the one saved first comes first (same tie rule as M15).

In [ ]:
real = pd.DataFrame({
    "source_id": [f"L-{i:03d}" for i in df["id"]],
    "description": df["description"],
    "amount": df["amount"],
    "category": df["category"],
    "note": df["note"].fillna(""),
    "status": "current",
    "authority": "student_confirmed",
    "scope": "student",
    "effective_date": df["date"],
    "owner": "student",
})
real.loc[real["source_id"] == "L-056", "effective_date"] = "2026-07-30"  # the corrected version was saved on 7/30
real.loc[real["source_id"] == "L-199", "scope"] = ""                       # injected defect: blank scope


def rec(**kw):
    return pd.DataFrame([kw])


RM_001 = rec(source_id="RM-001", description="VENMO PAYMENT 4410982231 WEB ID: 3264681992", amount=-412.50,
             category="Rent & Utilities", note="roommate's rent share", status="current",
             authority="student_confirmed", scope="roommate", effective_date="2026-07-01", owner="roommate")
V_056a = rec(source_id="V-056a", description="AMAZON PRIME*VP2Y93EMV AMZN.COM/BILL WA", amount=-7.49,
             category="Shopping", note="first label, changed later", status="superseded",
             authority="student_confirmed", scope="student", effective_date="2026-07-26", owner="student")
BK_039 = rec(source_id="BK-039", description="PURCHASE AUTHORIZED ON 07/17 PLANET FIT CLUB FEES TROY NY CARD 4821",
             amount=-15.00, category="Fees", note="bank category: Fees & Charges", status="current",
             authority="", scope="student", effective_date="2026-07-17", owner="nobody (bank import)")


def insert_before(frame, before_id, new_row):
    i = frame.index[frame["source_id"] == before_id][0]
    return pd.concat([frame.loc[: i - 1], new_row, frame.loc[i:]], ignore_index=True)


memory = pd.concat([RM_001, real], ignore_index=True)  # roommate record saved July 1, before everything else
memory = insert_before(memory, "L-039", BK_039)         # bank category saved at import, before the student's review
memory = insert_before(memory, "L-056", V_056a)         # first version saved before the corrected one
memory["text"] = memory["description"].map(normalize)

memory.to_csv("label_memory_snapshot.csv", index=False)  # save the exact snapshot we tested, for the worksheet
print(len(memory), "records in the label memory (211 real + 3 injected; 1 real record has a blank field)")
memory[memory["source_id"].isin(["RM-001", "L-007", "BK-039", "L-039", "V-056a", "L-056", "L-199"])][
    ["source_id", "text", "amount", "category", "status", "authority", "scope", "effective_date", "owner"]]

214 records in the label memory (211 real + 3 injected; 1 real record has a blank field)


,source_id,text,amount,category,status,authority,scope,effective_date,owner
0,RM-001,VENMO PAYMENT,-412.50,Rent & Utilities,current,student_confirmed,roommate,2026-07-01,roommate
7,L-007,VENMO PAYMENT,-25.94,Transfers,current,student_confirmed,student,2026-07-04,student
39,BK-039,PLANET FIT CLUB FEES TROY NY,-15.00,Fees,current,,student,2026-07-17,nobody (bank import)
40,L-039,PLANET FIT CLUB FEES TROY NY,-15.00,Health,current,student_confirmed,student,2026-07-17,student
57,V-056a,AMAZON PRIME * AMZN.COM/BILL WA,-7.49,Shopping,superseded,student_confirmed,student,2026-07-26,student
58,L-056,AMAZON PRIME * AMZN.COM/BILL WA,-7.49,Subscriptions,current,student_confirmed,student,2026-07-30,student
201,L-199,HUDSON VALLEY URGENT CARE TROY NY,-35.00,Health,current,student_confirmed,,2026-09-23,student


## Step 3. The eligibility rule (ONE rule)

> **A record may be used as evidence only if `status == current` AND `authority == student_confirmed` AND `scope == student`.**
> A blank field counts as "no" (if we can't tell, we don't trust it).

- **Who verifies the fields:** the app sets `status` when the student edits a label, and `scope` from the account the charge came from. `authority` is set to `student_confirmed` only when the student clicks to confirm or change a label.
- **Who can change them:** the student (their own records). Nobody can mark a bank import as confirmed except by confirming it.
- **What would make the rule unsafe:** a record wrongly marked `student_confirmed` (it gets trusted), or a good record with a missing field (it gets thrown away).

Then the **similarity search** is exactly M15's: TF-IDF on 3–5 letter pieces, cosine similarity, and a minimum score of **0.50** to count as evidence at all.

- **Condition A** = rank all records by similarity, take the top one (no rule).
- **Condition B** = the *same* ranking, but skip ineligible records; take the top *eligible* one. Log every record we skipped and why.

In [ ]:
MIN_SIM = 0.50
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
memory_vectors = vectorizer.fit_transform(memory["text"])


def why_ineligible(r) -> str:
    # Return '' if the record passes the rule, otherwise the reason it fails
    reasons = []
    if r["status"] != "current":
        reasons.append(f"status={r['status'] or 'blank'}")
    if r["authority"] != "student_confirmed":
        reasons.append(f"authority={r['authority'] or 'blank'}")
    if r["scope"] != "student":
        reasons.append(f"scope={r['scope'] or 'blank'}")
    return ", ".join(reasons)


memory["excluded_because"] = memory.apply(why_ineligible, axis=1)
print("Records the rule excludes:")
print(memory.loc[memory["excluded_because"] != "", ["source_id", "text", "category", "excluded_because"]].to_string(index=False))


def rank(description: str) -> pd.DataFrame:
    q = vectorizer.transform([normalize(description)])
    scores = (q @ memory_vectors.T).toarray()[0]
    order = np.argsort(-scores, kind="stable")  # ties keep the order records were saved
    out = memory.iloc[order].copy()
    out.insert(1, "similarity", scores[order].round(2))
    return out


def pick(ranked: pd.DataFrame, use_rule: bool):
    # Return (top record or None, list of skipped (id, reason) above it)
    skipped = []
    for _, r in ranked.iterrows():
        if r["similarity"] < MIN_SIM:
            return None, skipped
        if use_rule and r["excluded_because"]:
            skipped.append((r["source_id"], r["excluded_because"]))
            continue
        return r, skipped
    return None, skipped

Records the rule excludes:
source_id                              text         category  excluded_because
   RM-001                     VENMO PAYMENT Rent & Utilities    scope=roommate
   BK-039      PLANET FIT CLUB FEES TROY NY             Fees   authority=blank
   V-056a   AMAZON PRIME * AMZN.COM/BILL WA         Shopping status=superseded
    L-199 HUDSON VALLEY URGENT CARE TROY NY           Health       scope=blank


## Step 4. The five test cases

Each is a new October charge. "Expected" is what *should* happen, written before running — that's our prediction.

| # | Handout type | October charge | Correct category | Expected behaviour of the rule |
|---|---|---|---|---|
| 1 | Normal eligible | ALDI $58.14 | Groceries | Nothing to exclude. A and B should agree. |
| 2 | Archived / stale | Amazon Prime $7.49 | Subscriptions | Exclude `V-056a` (superseded); use the student's corrected label. |
| 3 | Wrong scope | Venmo payment $27.50 | Transfers | Exclude `RM-001` (roommate's account). |
| 4 | Missing / uncertain authority | Planet Fitness $15.00 | Health | Exclude `BK-039` (bank auto-category, authority blank). |
| 5 | Valid source over-filtered | Hudson Valley Urgent Care $35.00 (follow-up visit) | Health | `L-199` is a good record but its scope is blank, so the rule throws it out. Is "ask the student" an OK result? |

In [ ]:
CASES = [
    ("1 Normal eligible", "CHECKCARD 1006 ALDI 76012 LATHAM NY 55102938471", -58.14, "Groceries"),
    ("2 Archived/stale", "AMAZON PRIME*R82KD1LQ0 AMZN.COM/BILL WA", -7.49, "Subscriptions"),
    ("3 Wrong scope", "VENMO PAYMENT 6173320045 WEB ID: 3264681992", -27.50, "Transfers"),
    ("4 Missing authority", "POS PURCHASE PLANET FIT CLUB FEES TROY NY", -15.00, "Health"),
    ("5 Over-filtered", "HUDSON VALLEY URGENT CARE TROY NY", -35.00, "Health"),
]
SHOW = ["source_id", "similarity", "text", "category", "status", "authority", "scope", "excluded_because"]

rows = []
for case, desc, amt, truth in CASES:
    ranked = rank(desc)
    a, _ = pick(ranked, use_rule=False)
    b, skipped = pick(ranked, use_rule=True)
    print("=" * 120)
    print(f"CASE {case}:  {desc}  ${abs(amt):.2f}   correct category: {truth}")
    print(ranked.head(5)[SHOW].to_string(index=False))
    a_txt = f"{a['source_id']} -> {a['category']}" if a is not None else "no record above 0.50 -> ASK STUDENT"
    b_txt = f"{b['source_id']} -> {b['category']}" if b is not None else "no eligible record above 0.50 -> ASK STUDENT"
    print(f"\n  A (similarity only):     {a_txt}")
    print(f"  B (similarity + rule):   {b_txt}")
    print(f"  B skipped: {skipped if skipped else 'nothing'}")
    rows.append({"case": case, "correct": truth,
                 "A top ID": a["source_id"] if a is not None else "-",
                 "A answer": a["category"] if a is not None else "ASK STUDENT",
                 "B top ID": b["source_id"] if b is not None else "-",
                 "B answer": b["category"] if b is not None else "ASK STUDENT",
                 "B excluded": "; ".join(f"{i} ({why})" for i, why in skipped) or "-"})

results = pd.DataFrame(rows)

CASE 1 Normal eligible:  CHECKCARD 1006 ALDI 76012 LATHAM NY 55102938471  $58.14   correct category: Groceries
source_id  similarity             text  category  status         authority   scope excluded_because
    L-012        1.00   ALDI LATHAM NY Groceries current student_confirmed student                 
    L-116        1.00   ALDI LATHAM NY Groceries current student_confirmed student                 
    L-151        1.00   ALDI LATHAM NY Groceries current student_confirmed student                 
    L-011        0.46 TARGET LATHAM NY Groceries current student_confirmed student                 
    L-049        0.46 TARGET LATHAM NY  Shopping current student_confirmed student                 

  A (similarity only):     L-012 -> Groceries
  B (similarity + rule):   L-012 -> Groceries
  B skipped: nothing
CASE 2 Archived/stale:  AMAZON PRIME*R82KD1LQ0 AMZN.COM/BILL WA  $7.49   correct category: Subscriptions
source_id  similarity                            text      category   

### What happened, in plain words

- **Case 1:** the top record is `L-012`, a real ALDI label. Nothing was excluded. **The rule costs nothing when the data is clean.**
- **Case 2:** four Amazon Prime records tie at 1.00. Similarity can't tell them apart, so A takes the one saved first: `V-056a`, the **old** label the student already fixed → **Shopping, wrong** (handout code **S2 stale status**). B skips it because `status=superseded` and uses `L-056` → **Subscriptions, correct**.
- **Case 3:** all Venmo records clean to the same text `VENMO PAYMENT`. A takes `RM-001`, the **roommate's** rent payment → **Rent & Utilities, wrong** (**S3 wrong scope**). B skips it (`scope=roommate`) → `L-007`, **Transfers, correct**.
- **Case 4:** A takes `BK-039`, the **bank's** guess → **Fees, wrong** (it's the gym; "CLUB FEES" fooled the bank the same way it fooled our A2 keyword rule). B skips it (`authority=blank`) → `L-039`, **Health, correct**.
- **Case 5:** A takes `L-199` → **Health, correct.** B throws `L-199` away because its `scope` is blank, and the next records are our *paychecks* (`HUDSON DINING ... PAYROLL`, 0.19) — below 0.50, so B says **"ask the student."** That's a **false exclusion (S4 over-filtering)**. It's a *safe* failure (the student just gets asked one question), but it shows the rule is only as good as the fields. Without the 0.50 cutoff, B would have answered **Income**, which is much worse.

In [ ]:
results

,case,correct,A top ID,A answer,B top ID,B answer,B excluded
0,1 Normal eligible,Groceries,L-012,Groceries,L-012,Groceries,-
1,2 Archived/stale,Subscriptions,V-056a,Shopping,L-056,Subscriptions,V-056a (status=superseded)
2,3 Wrong scope,Transfers,RM-001,Rent & Utilities,L-007,Transfers,RM-001 (scope=roommate)
3,4 Missing authority,Health,BK-039,Fees,L-039,Health,BK-039 (authority=blank)
4,5 Over-filtered,Health,L-199,Health,-,ASK STUDENT,L-199 (scope=blank)


## Step 5. The handout's evidence sentence, generated for each case

In [ ]:
by_id = memory.set_index("source_id")
for r in rows:
    if r["B excluded"] != "-":
        first_id = r["B excluded"].split(" (")[0]
        reason = by_id.loc[first_id, "excluded_because"]
        verdict = "justified" if r["B answer"] == r["correct"] else "NOT justified (a good record was lost)"
        print(f"For case {r['case'][0]}, source {first_id} ranked 1st in A, but B excluded it because {reason} "
              f"under the eligibility rule. This was {verdict}: A said {r['A answer']}, B said {r['B answer']}, "
              f"correct is {r['correct']}.")
    else:
        print(f"For case {r['case'][0]}, source {r['A top ID']} ranked 1st in A and B kept it (passes the rule). "
              f"Both said {r['A answer']}.")

a_ok = (results["A answer"] == results["correct"]).sum()
b_ok = (results["B answer"] == results["correct"]).sum()
print(f"\nCorrect: A {a_ok}/5, B {b_ok}/5.  Unsafe inclusions avoided by B: 3/3 stress cases (2, 3, 4).  "
      f"False exclusions: 1 (case 5, ended in 'ask the student').")

For case 1, source L-012 ranked 1st in A and B kept it (passes the rule). Both said Groceries.
For case 2, source V-056a ranked 1st in A, but B excluded it because status=superseded under the eligibility rule. This was justified: A said Shopping, B said Subscriptions, correct is Subscriptions.
For case 3, source RM-001 ranked 1st in A, but B excluded it because scope=roommate under the eligibility rule. This was justified: A said Rent & Utilities, B said Transfers, correct is Transfers.
For case 4, source BK-039 ranked 1st in A, but B excluded it because authority=blank under the eligibility rule. This was justified: A said Fees, B said Health, correct is Health.
For case 5, source L-199 ranked 1st in A, but B excluded it because scope=blank under the eligibility rule. This was NOT justified (a good record was lost): A said Health, B said ASK STUDENT, correct is Health.

Correct: A 2/5, B 4/5.  Unsafe inclusions avoided by B: 3/3 stress cases (2, 3, 4).  False exclusions: 1 (case 5, en

## Step 6 (extra). What the rule *can't* see: false authority (S1)

The rule trusts any record marked `student_confirmed`. But in A2 we planned to **auto-accept the LLM's label when it says "sure"** and only ask the student when it's unsure. If those auto-accepted labels get saved as `student_confirmed`, the rule will trust LLM guesses as if the student checked them.

How much would that matter? We use the real A2 run (`B1_small.txt`, Claude Haiku 5.5 on the September statement) to count how many "sure" labels were wrong. This is a stress test of the *metadata*, not of the rule: we run both A and B on a snapshot where September's labels are replaced by the LLM's "sure" answers.

In [ ]:
b1_path, truth_path = find("B1_small.txt"), find("september_truth.csv")
if b1_path and truth_path:
    txt = open(b1_path).read()
    llm = {int(m.group(1)): (m.group(2).strip(), m.group(3).lower())
           for m in re.finditer(r"^\s*(\d+)\s*\|\s*([^|\n]+?)\s*\|\s*(sure|unsure)\b", txt, re.M | re.I)}
    truth = pd.read_csv(truth_path)
    truth["llm"] = truth["line"].map(lambda l: llm[l][0])
    truth["sure"] = truth["line"].map(lambda l: llm[l][1])
    sure = truth[truth["sure"] == "sure"]
    bad = sure[sure["llm"] != sure["category"]]
    print(f"LLM said 'sure' on {len(sure)} of {len(truth)} September charges; {len(bad)} of those {'was' if len(bad) == 1 else 'were'} wrong:")
    print(bad[["id", "description", "amount", "category", "llm", "note"]].to_string(index=False))

    # Altered snapshot: auto-accepted 'sure' labels overwrite September and are marked student_confirmed
    altered = memory.copy()
    for _, t in sure.iterrows():
        sel = altered["source_id"] == f"L-{t['id']:03d}"
        altered.loc[sel, "category"] = t["llm"]
        altered.loc[sel, "note"] = "auto-accepted LLM label"
    stress = "PURCHASE AUTHORIZED ON 10/12 APPLE.COM/BILL CUPERTINO CA CARD 4821"  # $0.99 one-off app purchase
    q = vectorizer.transform([normalize(stress)])
    sc = (q @ vectorizer.transform(altered["text"]).T).toarray()[0]
    top = altered.iloc[np.argsort(-sc, kind="stable")[:5]].copy()
    top.insert(1, "similarity", sc[np.argsort(-sc, kind="stable")[:5]].round(2))
    print("\nOctober Apple $0.99 app purchase on the altered snapshot (all of these pass the rule):")
    print(top[["source_id", "similarity", "text", "amount", "category", "note", "authority"]].to_string(index=False))
else:
    print("B1_small.txt / september_truth.csv not found; skipping this section.")

LLM said 'sure' on 59 of 68 September charges; 1 of those was wrong:
 id                                                        description  amount      category           llm                 note
167 PURCHASE AUTHORIZED ON 09/10 APPLE.COM/BILL CUPERTINO CA CARD 4821   -0.99 Entertainment Subscriptions one-off app purchase

October Apple $0.99 app purchase on the altered snapshot (all of these pass the rule):
source_id  similarity                        text  amount      category                      note         authority
    L-040         1.0 APPLE.COM/BILL CUPERTINO CA   -2.99 Subscriptions         iCloud+ 50GB plan student_confirmed
    L-073         1.0 APPLE.COM/BILL CUPERTINO CA   -4.99 Entertainment in-app purchase in a game student_confirmed
    L-107         1.0 APPLE.COM/BILL CUPERTINO CA   -2.99 Subscriptions         iCloud+ 50GB plan student_confirmed
    L-167         1.0 APPLE.COM/BILL CUPERTINO CA   -0.99 Subscriptions   auto-accepted LLM label student_confirmed
    L-1

Only 1 of the LLM's 59 "sure" labels was wrong (Apple $0.99 app purchase → Subscriptions), and the rule **can't catch it**: after auto-accept, `L-167` says `student_confirmed` but the student never saw it. Apple is a mixed store, so this record is now "trusted" evidence that $0.99 Apple charges are Subscriptions. In M15, the k = 5 answer step safely asked the student about a $0.99 Apple charge (no past charge was close in amount). On this snapshot it would find `L-167` at exactly $0.99 and confidently answer **Subscriptions — wrong**, citing a record that *looks* student-confirmed. Fix: save auto-accepted labels as `authority=llm_auto`, not `student_confirmed`, and decide separately whether the rule allows them.

## Summary for the worksheet

- **Filtering improved:** the 3 stress cases where a tempting record ranked first (stale, wrong scope, missing authority): A was wrong 3 times, B right 3 times.
- **Filtering worsened:** case 5 — one good record with a blank field was thrown out, and B fell back to "ask the student".
- **Not changed:** the normal case (1).
- **Biggest unresolved risk:** false authority. The rule trusts the `authority` field, and our own A2 design (auto-accept "sure" LLM labels) could fill it wrongly.

The filled-in worksheet is `CSCI4150_Hybrid_AI_Apply_Evaluate_Project_Handout.docx` in this folder.